In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
%run ../1_setup/utils

In [0]:
## Enter manually the S3 uri to access data.

dbutils.widgets.text("data_source", "s3 uri", "Data Source")
# e.g. s3://bucket_name/customers/*.csv
data_source = dbutils.widgets.get("data_source")
print(data_source)

## Bronze


In [0]:
df_bronze = spark.read.csv(data_source, header=True, inferSchema=True)
display(df_bronze.limit(5))

In [0]:
df_bronze = df_bronze.withColumn('ingested_at', F.current_timestamp())\
    .withColumn('_source_file', F.col('_metadata.file_path'))\
    .withColumn('_file_size', F.col('_metadata.file_size'))\
    .withColumn('customer_id', F.col('customer_id').cast(StringType()))

df_bronze.printSchema()

In [0]:
df_bronze.write.mode('overwrite')\
    .format('delta')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{bronze_schema}.dim_customers')

## Silver

In [0]:
df_bronze = spark.table(f'{catalog}.{bronze_schema}.dim_customers')
display(df_bronze.limit(10))

In [0]:
df_silver = df_bronze.withColumn('customer_name', F.trim(F.col("customer_name")))\
    .withColumn('city', F.trim(F.col("city")))

In [0]:
df_silver.select('customer_name').distinct().show(truncate=False)

In [0]:
df_silver = df_silver.withColumn('customer_name', F.initcap(F.col('customer_name')))
df_silver.select('customer_name').distinct().show(truncate=False)

In [0]:
df_silver.select('city').distinct().show()

In [0]:
city_fix = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

df_silver = df_silver.replace(city_fix, subset=['city'])
df_silver.select('city').distinct().show()

In [0]:
df_silver.filter(df_silver.city.isNull()).show()

In [0]:
# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(key, value) for key, value in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

df_silver = df_silver.join(df_fix, on='customer_id', how='left')\
    .withColumn('city', F.coalesce(F.col('fixed_city'), F.col('city')))\
    .drop('fixed_city')
df_silver.filter(df_silver.city.isNull()).show()

In [0]:
df_silver.dropDuplicates(['customer_id'])

In [0]:
df_silver.write.mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{silver_schema}.dim_customers')

## Gold

In [0]:
df_silver = spark.table(f'{catalog}.{silver_schema}.dim_customers')

In [0]:

# Standardizing customer attributes to match Parent Company data model

df_gold = df_silver.withColumn('customer_name', F.concat(F.col('customer_name'),F.lit('-'),F.col('city')))\
    .drop('city')\
    .withColumnRenamed('customer_id','customer_code')\
    .withColumnRenamed('customer_name','customer')\
    .withColumn('market', F.lit('India'))\
    .withColumn('platform', F.lit('Sports Bar'))\
    .withColumn('channel', F.lit('Acquisition'))

display(df_gold.limit(10))


In [0]:
df_gold.write.mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{gold_schema}.child_dim_customers')

## Merging Child Company with Parent

In [0]:

%sql
MERGE INTO fmcg.gold.dim_customers AS target
USING fmcg.gold.child_dim_customers AS source
ON target.customer_code = source.customer_code
WHEN MATCHED THEN
  UPDATE SET
    customer = source.customer,
    market = source.market,
    platform = source.platform,
    channel = source.channel
WHEN NOT MATCHED THEN
  INSERT (customer_code, customer, market, platform, channel)
  VALUES (source.customer_code, source.customer, source.market, source.platform, source.channel)